<a href="https://colab.research.google.com/github/beingAnujChaudhary/DSFS-Joel-Grus/blob/main/notebooks/chapter_06_probability/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/DSFS-Joel-Grus.git

# Move into repository
%cd /content/DSFS-Joel-Grus

# Move into Chapter 6 folder
%cd notebooks/chapter_06_probability

Mounted at /content/drive
Cloning into 'DSFS-Joel-Grus'...
remote: Enumerating objects: 329, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 329 (delta 224), reused 189 (delta 139), pack-reused 0 (from 0)
Receiving objects: 100% (329/329), 2.40 MiB | 10.79 MiB/s, done.
Resolving deltas: 100% (224/224), done.
/content/DSFS-Joel-Grus
/content/DSFS-Joel-Grus/notebooks/chapter_06_probability


---

# Chapter 6: Probability — Experiments

## Purpose

This notebook is your playground for exploring probability concepts beyond the book's examples.

The goal is to:
- Simulate counter-intuitive conditional probability paradoxes
- Explore how base rates dramatically shift Bayesian posterior probabilities
- Visualize the Central Limit Theorem (CLT) across different underlying distributions
- Build intuition for randomness and uncertainty through simulation
- Connect probability to machine learning applications

**Rule:**
> Experiment with randomness instead of memorizing formulas. Do not worry about breaking things — experimentation is the point.

---



In [2]:
## Environment Setup

import random
import math
import matplotlib.pyplot as plt
from collections import Counter

# Reuse core functions from Chapter 6
def normal_cdf(x, mu=0, sigma=1):
    return (1 + math.erf((x - mu) / math.sqrt(2) / sigma)) / 2

def inverse_normal_cdf(p, mu=0, sigma=1, tolerance=0.00001):
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p, tolerance=tolerance)
    low_z, low_p = -10.0, 0
    hi_z, hi_p   =  10.0, 1
    while hi_z - low_z > tolerance:
        mid_z = (low_z + hi_z) / 2
        mid_p = normal_cdf(mid_z)
        if mid_p < p:
            low_z, low_p = mid_z, mid_p
        elif mid_p > p:
            hi_z, hi_p = mid_z, mid_p
        else:
            break
    return mid_z

def normal_pdf(x, mu=0, sigma=1):
    sqrt_two_pi = math.sqrt(2 * math.pi)
    return (math.exp(-(x - mu) ** 2 / 2 / sigma ** 2) / (sqrt_two_pi * sigma))

print("✅ Probability experiments environment ready!")

✅ Probability experiments environment ready!


---

## Experiment 1 — Single Coin Flip: How Random is Randomness?

```python
# Run this cell multiple times to see different results
coin = random.choice(["Heads", "Tails"])
print(f"Result: {coin}")

# The result can change each time because it's genuinely random
# Each flip is an independent event with P(Heads) = 0.5
```

### 💡 Reflection
- Why can the result change every time you run it?
- If you run it 10 times and get 7 Heads, is the coin unfair?


---

## Experiment 2 — Many Coin Flips & Convergence

```python
heads = 0
tails = 0

for _ in range(100):
    flip = random.choice(["Heads", "Tails"])
    if flip == "Heads":
        heads += 1
    else:
        tails += 1

print(f"After 100 flips: Heads = {heads}, Tails = {tails}")

# Try different numbers of flips
for n in [10, 100, 1000, 10000]:
    h = sum(1 for _ in range(n) if random.choice(["Heads", "Tails"]) == "Heads")
    print(f"After {n:>5} flips: {h/n:.3f} Heads ({h/n*100:.1f}%)")
```

### 💡 Reflection
- Why do results move closer to 50/50 as n increases?
- This is the **Law of Large Numbers** in action!


---

## Experiment 3 — Visualizing Convergence to Probability

```python
results = []
heads = 0

for i in range(1, 1001):
    flip = random.choice(["Heads", "Tails"])
    if flip == "Heads":
        heads += 1
    proportion_heads = heads / i
    results.append(proportion_heads)

plt.figure(figsize=(12, 5))
plt.plot(results, linewidth=1, alpha=0.8)
plt.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='True Probability (0.5)')
plt.title("Convergence to Probability (Law of Large Numbers)", fontsize=14, fontweight='bold')
plt.xlabel("Number of Flips")
plt.ylabel("Proportion of Heads")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final proportion after 1000 flips: {results[-1]:.3f}")
```

### 💡 Reflection
- Why does the line stabilize near 0.5 over time?
- Notice the early fluctuations — small samples can be misleading!



---

## Experiment 4 — Dice Simulation

```python
rolls = [random.randint(1, 6) for _ in range(1000)]

plt.figure(figsize=(10, 6))
plt.hist(rolls, bins=6, edgecolor='black', color='steelblue', alpha=0.8)
plt.title("Dice Roll Distribution (1000 rolls)", fontsize=14, fontweight='bold')
plt.xlabel("Dice Value")
plt.ylabel("Frequency")
plt.xticks([1, 2, 3, 4, 5, 6])
plt.grid(axis='y', alpha=0.3)
plt.show()

# Count frequencies
counts = Counter(rolls)
for face in sorted(counts.keys()):
    expected = 1000 / 6
    print(f"  Face {face}: {counts[face]} times (expected ~{expected:.0f}, diff: {counts[face] - expected:+.0f})")
```

### 💡 Reflection
- Why do all numbers appear roughly equally?
- The sample space is {1,2,3,4,5,6} with each having probability 1/6



---

## Experiment 5 — Even Number Probability (Event Simulation)

```python
even_count = 0
num_trials = 1000

for _ in range(num_trials):
    roll = random.randint(1, 6)
    if roll % 2 == 0:
        even_count += 1

probability = even_count / num_trials
print(f"P(even) by simulation: {probability:.3f}")
print(f"P(even) theoretically:  {3/6:.3f}")
print(f"Difference: {abs(probability - 0.5):.4f}")
```

### 💡 Reflection
- Why is the simulated probability close to 0.5?
- Favorable outcomes {2,4,6} out of 6 total → 3/6 = 0.5


---

## Experiment 6 — Independent Events

```python
# Demonstrate that coin flips are independent
print("Five pairs of coin flips:")
for i in range(5):
    flip_1 = random.choice(["Heads", "Tails"])
    flip_2 = random.choice(["Heads", "Tails"])
    print(f"  Pair {i+1}: {flip_1:>6} | {flip_2:<6}")

# Statistical verification
both_heads = 0
first_heads = 0

for _ in range(10000):
    f1 = random.choice(["Heads", "Tails"])
    f2 = random.choice(["Heads", "Tails"])
    if f1 == "Heads":
        first_heads += 1
        if f2 == "Heads":
            both_heads += 1

p_second_given_first = both_heads / first_heads
print(f"\nP(Heads on 2nd | Heads on 1st) = {p_second_given_first:.3f}")
print(f"P(Heads on 2nd) = 0.500")
print(f"→ Equal? {abs(p_second_given_first - 0.5) < 0.02} (confirms independence)")
```

### 💡 Reflection
- Does flip 1 influence flip 2? No — they're independent
- What would make events dependent? (e.g., drawing cards without replacement)


---

## Experiment 7 — Conditional Thinking

```python
# Prior probability
p_rain = 0.30
print(f"P(Rain) = {p_rain:.0%}")

# After seeing evidence (dark clouds)
p_rain_given_clouds = 0.70
print(f"P(Rain | Clouds) = {p_rain_given_clouds:.0%}")

print(f"\n🔍 Key idea: New evidence → Updated belief")
print(f"   This is Bayesian thinking — updating probabilities")
print(f"   when relevant information becomes available.")
```

### 💡 Reflection
- Why did the probability change from 30% to 70%?
- Conditional probability: P(A|B) ≠ P(A) when B provides relevant information


---

## Experiment 8 — Random Distributions

```python
random_numbers = [random.randint(1, 10) for _ in range(1000)]

plt.figure(figsize=(10, 6))
plt.hist(random_numbers, bins=10, edgecolor='black', color='coral', alpha=0.8)
plt.title("Random Number Distribution (Uniform 1-10)", fontsize=14, fontweight='bold')
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.xticks(range(1, 11))
plt.grid(axis='y', alpha=0.3)
plt.show()
```

### 💡 Reflection
- Why does the graph look roughly balanced?
- Uniform distribution: each value 1-10 has equal probability



---

## Experiment 9 — The Monty Hall Problem

A famous conditional probability puzzle: 3 doors, one car, two goats. After you pick, the host opens a goat door. Should you switch?

```python
def simulate_monty_hall(num_trials=10000):
    switch_wins = 0
    stay_wins = 0
    
    for _ in range(num_trials):
        doors = [0, 1, 2]
        car = random.choice(doors)
        initial_choice = random.choice(doors)
        
        # Host opens a door that is NOT the car and NOT the initial choice
        available_to_host = [d for d in doors if d != car and d != initial_choice]
        host_opens = random.choice(available_to_host)
        
        # Determine the switch door
        switch_choice = [d for d in doors if d != initial_choice and d != host_opens][0]
        
        if initial_choice == car:
            stay_wins += 1
        if switch_choice == car:
            switch_wins += 1
    
    return stay_wins / num_trials, switch_wins / num_trials

stay_pct, switch_pct = simulate_monty_hall(10000)

print("Monty Hall Problem Results (10,000 simulations):")
print(f"  Win if STAY:   {stay_pct:.3f} (~1/3)")
print(f"  Win if SWITCH: {switch_pct:.3f} (~2/3)")
print(f"\n🔍 Switching doubles your chances!")
print(f"   → The host's knowledge provides new information")
print(f"   → Conditional probability at work!")
```

### 💡 Try modifying:
- Change to 4 or 10 doors — how do the odds change?
- **Ask:** Why does the host's knowledge of the car's location matter?


---

## Experiment 10 — Bayes's Theorem: Base Rate Sensitivity

How does the posterior probability change as we vary the prior (base rate) and test accuracy?

```python
def bayes_theorem(p_prior, p_true_positive, p_false_positive):
    p_not_prior = 1 - p_prior
    p_positive = (p_true_positive * p_prior) + (p_false_positive * p_not_prior)
    return (p_true_positive * p_prior) / p_positive

# Vary priors and accuracies
priors = [0.0001, 0.001, 0.01, 0.05, 0.10]     # 1 in 10k to 1 in 10
accuracies = [0.90, 0.95, 0.99, 0.999]          # True positive rates

print(f"{'Prior':<15}", end="")
for acc in accuracies:
    print(f"  Acc={acc}", end="")
print("\n" + "-" * 65)

for prior in priors:
    print(f"{prior:<15.4f}", end="")
    for acc in accuracies:
        false_pos = 1 - acc
        posterior = bayes_theorem(prior, acc, false_pos)
        print(f"  {posterior:.4f}", end="")
    print()

print(f"\n🔍 Even with 99.9% accuracy, a rare disease (1/10,000)")
print(f"   yields only ~9% chance of actually having it!")
```

### 💡 Reflection
- Why do base rates matter so much?
- In ML: fraud detection (rare events) needs very low false positive rates



---

## Experiment 11 — The Central Limit Theorem on Skewed Data

The CLT works on any distribution — even highly skewed ones!

```python
# Generate a highly skewed underlying population (Exponential)
population = [random.expovariate(1) for _ in range(100000)]

def get_sample_means(pop, sample_size, num_samples):
    means = []
    for _ in range(num_samples):
        sample = random.sample(pop, sample_size)
        means.append(sum(sample) / sample_size)
    return means

# Test different sample sizes
sample_sizes = [1, 5, 30, 100]

plt.figure(figsize=(15, 10))
for i, n in enumerate(sample_sizes, 1):
    sample_means = get_sample_means(population, n, 5000)
    
    plt.subplot(2, 2, i)
    plt.hist(sample_means, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
    plt.title(f"Sample Size n = {n}", fontweight='bold')
    plt.xlabel("Sample Mean")
    plt.ylabel("Density")
    
    # Overlay theoretical Normal curve
    pop_mu = sum(population) / len(population)
    pop_sigma = math.sqrt(sum((x - pop_mu)**2 for x in population) / len(population))
    mu = pop_mu
    sigma = pop_sigma / math.sqrt(n)
    
    xs = [x / 100.0 for x in range(0, int(max(sample_means)*100))]
    ys = [normal_pdf(x, mu, sigma) for x in xs]
    plt.plot(xs, ys, 'r-', linewidth=2, label='Normal Approx')
    plt.legend()

plt.suptitle("CLT: Convergence to Normal from Skewed (Exponential) Data",
             y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔍 Even from highly skewed data, sample means become Normal!")
print("   This is why the Normal distribution is everywhere in statistics.")
```

### 💡 Try modifying:
- Use a Uniform or Pareto distribution — does CLT still hold?
- **Ask:** At what sample size n does the CLT approximation become "good enough"?



---

## Experiment 12 — Inverse Transform Sampling

How do computers generate Normal random numbers? They pass Uniform(0,1) through the Inverse CDF!

```python
# Generate 10,000 Uniform random numbers
uniform_samples = [random.random() for _ in range(10000)]

# Transform them into Normal random numbers using Inverse CDF
normal_samples = [inverse_normal_cdf(p, mu=50, sigma=15) for p in uniform_samples]

plt.figure(figsize=(12, 5))

# Plot generated Normal distribution
plt.subplot(1, 2, 1)
plt.hist(normal_samples, bins=50, density=True, alpha=0.7, color='lightgreen', edgecolor='black')
plt.title("Generated Normal(50, 15)\nvia Inverse Transform", fontweight='bold')
plt.xlabel("Value")
plt.ylabel("Density")

# Overlay true Normal PDF
xs = [x / 10.0 for x in range(0, 1000)]
ys = [normal_pdf(x, mu=50, sigma=15) for x in xs]
plt.plot(xs, ys, 'r-', linewidth=2, label='True Normal PDF')
plt.legend()

# Plot original Uniform samples
plt.subplot(1, 2, 2)
plt.hist(uniform_samples, bins=20, density=True, alpha=0.7, color='salmon', edgecolor='black')
plt.title("Original Uniform(0,1) Samples", fontweight='bold')
plt.xlabel("Value")
plt.ylabel("Density")
plt.axhline(y=1, color='red', linestyle='--', label='Uniform PDF')
plt.legend()

plt.tight_layout()
plt.show()

print("🔍 We built a Normal random number generator from scratch!")
print("   Uniform(0,1) → Inverse Normal CDF → Normal distribution")
print("   → This is how computers generate samples from any distribution!")
```

### 💡 Reflection
- How does this technique connect to modern generative AI (VAEs, diffusion models)?
- Why is the Inverse CDF method fundamental to probabilistic programming?


---

## Experiment 13 — Probability and Uncertainty in ML

```python
print("=" * 60)
print("PROBABILITY IN MACHINE LEARNING SYSTEMS")
print("=" * 60)

examples = [
    ("Spam Filter", "P(spam | email content) = 92%", "Not certain — just highly likely"),
    ("Recommendation", "P(user clicks | item) = 15%", "Enables ranking items by likelihood"),
    ("Medical AI", "P(disease | scan) = 87%", "Flags uncertain cases for human review"),
    ("Fraud Detection", "P(fraud | transaction) = 3%", "Rare events need careful thresholds"),
    ("Weather Forecast", "P(rain tomorrow) = 30%", "Probability, not certainty"),
]

for system, probability, interpretation in examples:
    print(f"\n📌 {system}")
    print(f"   Prediction: {probability}")
    print(f"   Meaning: {interpretation}")

print(f"\n{'=' * 60}")
print("KEY INSIGHT:")
print("  ML predictions are probabilistic, not perfectly certain.")
print("  → '92% spam' is more informative than just 'spam'")
print("  → Probabilities enable ranking, thresholding, and risk assessment")
```


---

## Experiment 14 — Probability in Real Life

```python
print("=" * 60)
print("PROBABILITY IN EVERYDAY DECISIONS")
print("=" * 60)

scenarios = [
    ("Weather forecast", "P(rain) = 30%", "Bring an umbrella or not?"),
    ("Flight booking", "P(delay) = 15%", "Allow extra connection time?"),
    ("Exam preparation", "P(passing | study) = 85%", "How much to study?"),
    ("Medical screening", "P(disease | positive) ≈ 1%", "Should you worry?"),
    ("Investment", "P(return > 10%) = 60%", "Is the risk worth it?"),
]

for scenario, probability, decision in scenarios:
    print(f"\n📌 {scenario}")
    print(f"   {probability}")
    print(f"   Decision: {decision}")

print(f"\n{'=' * 60}")
print("🔍 Probability helps us make decisions under uncertainty.")
print("   It doesn't eliminate uncertainty — it quantifies it.")
print("   This is the foundation of data-driven decision making.")
```


---

## 🔑 Key Takeaways

| Experiment | Insight | ML/Data Science Connection |
|-----------|---------|----------------------------|
| Coin flips & LLN | Frequencies converge to true probability with more trials | Sample size matters for reliable model evaluation |
| Convergence plot | Early samples are noisy; stabilization requires many trials | Small test sets give unreliable accuracy estimates |
| Dice simulation | Uniform distributions show equal likelihood across outcomes | Understanding data distributions before modeling |
| Independent events | One event doesn't affect another when independent | Naive Bayes assumption in classification |
| Conditional thinking | New evidence updates probabilities | Bayesian updating, online learning |
| Monty Hall | Host's knowledge provides information that changes odds | Importance of conditioning on all available information |
| Base rate sensitivity | Posterior depends heavily on prior when events are rare | Imbalanced classification, fraud detection thresholds |
| CLT on skewed data | Sample means converge to Normal regardless of population shape | A/B testing, confidence intervals, regression assumptions |
| Inverse transform | Map Uniform(0,1) to any distribution via Inverse CDF | Generating synthetic data, weight initialization, VAEs |
| ML applications | Predictions are probabilistic, not certain | Ranking, thresholding, risk quantification |

---

## 📝 Final Reflections

### Questions to think about

1. Why does randomness matter in data science?
2. Why do probabilities stabilize with more data (Law of Large Numbers)?
3. Why does new evidence change probabilities (conditional probability)?
4. Why are ML predictions probabilistic rather than perfectly certain?
5. Where do you already use probability thinking in daily life?
6. How does the CLT enable simpler statistical analysis?
7. What's the most surprising probability behavior you observed?

### Biggest Takeaway

> Probability is not about certainty — it's about reasoning under uncertainty.

Machine learning systems rarely say "100% certain." Instead they say:
- "Likely spam (92% confidence)"
- "Probable fraud (87% probability)"
- "User might click (15% chance)"

This probabilistic mindset — quantifying uncertainty, updating beliefs with evidence, and making decisions under incomplete information — is essential for AI/ML engineering.

---

**Next Steps:** Pick one experiment and push it further — try the CLT with a Cauchy distribution (which has no mean!) and watch it fail

> Remember: Probability is the logic of science. Human intuition is notoriously bad at it; simulations and math are our only reliable guides.